# Visualiser les Boundary Tokens
Crée une page HTML avec tous les boundary tokens surlignés

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install transformers -q
print("Dependencies OK")

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

# Load corpus
print("[1/4] Load corpus...")
with open('data/processed/kitab_uqala_reference_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"      {len(text):,} chars")

# Load boundary tokens
print("\n[2/4] Load boundary tokens...")
with open('results/camelbert_boundary_tokens_clean.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
boundary_indices = set(data['boundary_indices'])
print(f"      {len(boundary_indices):,} boundary tokens")

# Tokenize
print("\n[3/4] Tokenize corpus...")
tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/bert-base-arabic-camelbert-msa')
tokens = tokenizer.tokenize(text)
print(f"      {len(tokens):,} tokens total")

In [ ]:
# Generate HTML
print("\n[4/4] Generate HTML...")

html = """<!DOCTYPE html>
<html dir="rtl" lang="ar">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Visualisation Boundary Tokens - Kitab Uqala</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Naskh+Arabic:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {
            direction: rtl;
            font-family: 'Noto Naskh Arabic', Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
            line-height: 2;
        }
        .container {
            max-width: 1000px;
            margin: 0 auto;
            background-color: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }
        h1 { text-align: center; color: #333; }
        .info { text-align: center; color: #666; font-size: 14px; margin: 20px 0; padding: 10px; background-color: #f0f0f0; border-radius: 5px; }
        .legend { text-align: center; margin: 20px 0; padding: 15px; background-color: #fafafa; border-radius: 5px; }
        .text-container { background-color: white; padding: 20px; border: 1px solid #ddd; border-radius: 5px; text-align: justify; font-size: 16px; }
        .boundary-token { background-color: rgba(255, 165, 0, 0.3); border-left: 3px solid #ff8c00; padding: 2px 4px; margin: 0 2px; border-radius: 2px; }
        .stats { margin-top: 20px; padding: 15px; background-color: #f0f0f0; border-radius: 5px; font-size: 14px; }
        .stat-value { font-weight: bold; color: #ff8c00; }
        .footer { text-align: center; color: #999; font-size: 12px; margin-top: 30px; padding-top: 20px; border-top: 1px solid #ddd; }
    </style>
</head>
<body>
    <div class="container">
        <h1>📚 Boundary Tokens - Kitab Uqala</h1>
        <div class="info">Tokens en <span style="background-color: rgba(255, 165, 0, 0.3); padding: 2px 4px;">orange</span> = boundary tokens</div>
        <div class="legend"><strong>Légende:</strong> Orange = Boundary Token (classifié comme isnad)</div>
        <div class="text-container">"""

# Add tokens
for token_idx, token in enumerate(tokens):
    if token_idx in boundary_indices:
        html += f'<span class="boundary-token">{token}</span>'
    else:
        html += token
    if not token.startswith('##'):
        html += ' '

html += f"""</div>
        <div class="stats">
            <p><strong>Statistiques:</strong></p>
            <p>Corpus: <span class="stat-value">{len(text):,}</span> chars</p>
            <p>Tokens: <span class="stat-value">{len(tokens):,}</span></p>
            <p>Boundary tokens: <span class="stat-value">{len(boundary_indices):,}</span></p>
            <p>Pourcentage: <span class="stat-value">{100 * len(boundary_indices) / len(tokens):.2f}%</span></p>
        </div>
        <div class="footer">
            <p>CAMeL-BERT Visualization - 2026-04-21</p>
        </div>
    </div>
</body>
</html>"""

# Save
with open('results/visualization_boundary_tokens.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f"✓ HTML generated: results/visualization_boundary_tokens.html")
print(f"\nStats:")
print(f"  Boundary tokens: {len(boundary_indices):,}")
print(f"  Percentage: {100 * len(boundary_indices) / len(tokens):.2f}%")

In [ ]:
# Download
from google.colab import files
files.download('results/visualization_boundary_tokens.html')
print("Downloaded!")